# Preprocesado y Feature Engineering
En este notebook transformaremos el dataset ya saneado de transacciones individuales en una serie temporal diaria, enriqueciendolo mediante ingeniria de variables.

In [7]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

# [DOC] Realizamos la carga de datos, la creación de la variable Sales y extraemos solo la fecha par agrupar todas las transacciones de un mismo día
df_processed = pd.read_csv('../../data/interim/data_sanitized.csv', parse_dates=['InvoiceDate'])

df_processed['Sales'] = df_processed['Quantity'] * df_processed['UnitPrice']

df_processed['Date'] = df_processed['InvoiceDate'].dt.normalize()

df_daily_sales = df_processed.groupby('Date').agg({
    'Sales': 'sum',
    'InvoiceNo': 'nunique',
    'CustomerID': 'nunique'
}).reset_index()

df_daily_sales.rename(columns={
    'InvoiceNo': 'TransactionCount', 
    'UniqueCustomers': 'UniqueCustomers'
}, inplace=True)

df_daily_sales.sort_values('Date', inplace=True)

In [ ]:
# [DOC] generamos variables temporales básicas, variables de memoria y ventanas móviles

# [DOC] Variables Temporales Básicas
df_daily_sales['DayOfWeek'] = df_daily_sales['Date'].dt.dayofweek
df_daily_sales['Month'] = df_daily_sales['Date'].dt.month
df_daily_sales['IsWeekend'] = df_daily_sales['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# [DOC] Variables de Memoria (Lags)
# No solo el día anterior (Lag 1), sino el mismo día de la semana pasada (Lag 7)
df_daily_sales['Sales_Lag1'] = df_daily_sales['Sales'].shift(1)
df_daily_sales['Sales_Lag7'] = df_daily_sales['Sales'].shift(7)

# [DOC] Ventanas Móviles (Rolling Windows)
# Promedio de ventas de los últimos 7 días (suaviza picos aislados)
df_daily_sales['Rolling_Mean_7'] = df_daily_sales['Sales'].shift(1).rolling(window=7).mean()

df_daily_sales.dropna(inplace=True)

In [ ]:
# [DOC] Aplicamos One-Hot Encoding para el día de la semana y separamos el conjunto de entrenamiento y test antes de escalar

df_daily_sales = pd.get_dummies(df_daily_sales, columns=['DayOfWeek'], prefix='Day')

split_date = pd.to_datetime('2011-11-09')

df_train = df_daily_sales[df_daily_sales['Date'] < split_date].copy()
df_test = df_daily_sales[df_daily_sales['Date'] >= split_date].copy()

print(f"[INFO] Registros Entrenamiento: {len(df_train)}")
print(f"[INFO] Registros Test: {len(df_test)}")

[INFO] Registros Entrenamiento: 271
[INFO] Registros Test: 27


In [10]:
# [DOC] Escalamos las variables numéricas para que todas tengan el mismo peso en el modelo.

features_to_scale = ['Sales_Lag1', 'Sales_Lag7', 'Rolling_Mean_7', 'TransactionCount']
scaler = StandardScaler()

df_train[features_to_scale] = scaler.fit_transform(df_train[features_to_scale])
df_test[features_to_scale] = scaler.transform(df_test[features_to_scale])

In [11]:
# [DOC] Realizamos el guardado de los datos procesados

output_dir = '../../data/processed'

if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    print(f"[INFO] Carpeta creada con éxito: {output_dir}")

df_train.to_csv(os.path.join(output_dir, 'train_data.csv'), index=False)
df_test.to_csv(os.path.join(output_dir, 'test_data.csv'), index=False)

print(f"[SUCCESS] Archivos guardados correctamente en: {output_dir}")
display(df_train.head())

[INFO] Carpeta creada con éxito: ../../data/processed
[SUCCESS] Archivos guardados correctamente en: ../../data/processed


,Date,Sales,TransactionCount,CustomerID,Month,IsWeekend,Sales_Lag1,Sales_Lag7,Rolling_Mean_7,Day_0,Day_1,Day_2,Day_3,Day_4,Day_6
7,2010-12-09,30218.95,2.186918,88,12,0,1.527858,1.701851,1.523415,False,False,False,True,False,False
8,2010-12-10,27177.24,0.911706,53,12,0,1.031841,1.916561,1.388964,False,False,False,False,True,False
9,2010-12-12,16788.64,-0.629175,41,12,1,0.693646,0.145325,1.136129,False,False,False,False,False,True
10,2010-12-13,24829.71,0.486635,58,12,0,-0.461420,0.924451,1.008887,True,False,False,False,False,False
11,2010-12-14,26671.93,1.283643,71,12,0,0.432633,0.610659,0.909659,False,True,False,False,False,False
